## Agents and tools using LCEL

In [43]:
import langchain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

google_llm = ChatGoogleGenerativeAI(
    temperature=0, 
    model="gemini-2.0-flash", 
    api_key=google_api_key,
    max_tokens=200
)

openai_llm = ChatOpenAI(
    temperature=0, 
    model="gpt-4", 
    api_key=openai_api_key
)


In [44]:
from langchain.agents import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

pdf_loader_1 = PyPDFLoader(
    "./docs_for_rag/for_agents_lectures/wellarchitected-framework.pdf",
)

pdf_loader_2 = PyPDFLoader(
    "./docs_for_rag/for_agents_lectures/gzip.pdf",
)

text_loader = TextLoader(
    "./docs_for_rag/for_agents_lectures/coolie_english.txt",
)

pdf_1_docs = pdf_loader_1.load()
pdf_2_docs = pdf_loader_2.load()
text_docs = text_loader.load()

all_docs = pdf_1_docs + pdf_2_docs + text_docs

In [ ]:
print(len(pdf_1_docs))
print(len(pdf_2_docs))
print(len(text_docs))

# all_docs = text_docs

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(all_docs)

In [46]:
len(split_docs)

1476

In [47]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(split_docs, embeddings)

In [ ]:
results = await vectorstore.asimilarity_search("who is dahaa in coolie?")
results

In [ ]:
retriever = vectorstore.as_retriever()

results = retriever.invoke("who is dahaa in coolie?")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", """Always answer the question just by using the context provided and not from your knowledge.
        Context: {context}
        question: {input}
     
        Answer: 
     """),
     ("placeholder", "{agent_scratchpad}")
])


chain = {"context": retriever, "input": RunnablePassthrough()} | prompt | google_llm

# chain = {"context": RunnableLambda(lambda x: x["input"]) | retriever, "input": RunnableLambda(lambda x: x["input"])} | prompt | google_llm | StrOutputParser()

res = chain.invoke("What is gzip?")

res

#### Tool creation - RAG

In [ ]:
from langchain.agents import tool

@tool
def search_documents(input: str) -> str:
    """Search through documents to answer questions about AWS Well-Architected Framework, GZIP, and Coolie movie."""
    
    retriever = vectorstore.as_retriever()

    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough, RunnableLambda
    from langchain_core.output_parsers import StrOutputParser

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant"),
        ("human", """Always answer the question just by using the context provided and not from your knowledge.
            Context: {context}
            question: {input}
        
            Answer: 
        """),
        ("placeholder", "{agent_scratchpad}")
    ])


    chain = {"context": retriever, "input": RunnablePassthrough()} | prompt | openai_llm | StrOutputParser()

    rag_search_result = chain.invoke(input)

    return rag_search_result

##### Database agent - Using create_sql_agent

In [ ]:
import os
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain.agents import tool

db_url = os.getenv('DATABASE_URL')

db = SQLDatabase.from_uri(db_url) # db.run("SELECT * FROM users;")

agent_executor = create_sql_agent(db=db, llm=google_llm, verbose=True)

@tool
def search_database(input: str):
    """Search the users database to answer queries about user information.
    
    Use this tool when you need to find information about users such as:
    - User details (username, name, email, date of birth)
    - User account status (is_active)
    - User creation dates
    
    Examples of queries this tool can handle:
    - "What is the email of John Doe?"
    - "Find all active users"
    - "When was Alice's account created?"
    - "List all usernames in the database"
    
    Args:
        input: Natural language query about users
        
    Returns:
        str: Answer to the query based on the database
    """
    res = agent_executor.invoke({"input": input})
    return res



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input:usersI should query the schema of the users table to see what columns are available.
Action: sql_db_schema
Action Input: users
CREATE TABLE users (
	id SERIAL NOT NULL, 
	username VARCHAR(50) NOT NULL, 
	name VARCHAR(100), 
	date_of_birth DATE, 
	creation_date TIMESTAMP WITHOUT TIME ZONE DEFAULT now(), 
	email VARCHAR(100), 
	is_active BOOLEAN DEFAULT true, 
	CONSTRAINT users_pkey PRIMARY KEY (id), 
	CONSTRAINT users_username_key UNIQUE NULLS DISTINCT (username)
)

/*
3 rows from users table:
id	username	name	date_of_birth	creation_date	email	is_active
1	praveen12	Praveen Kumar	1990-05-21	2025-11-06 20:45:21.433560	praveen@example.com	True
2	alice88	Alice Johnson	1988-09-15	2025-11-13 20:45:21.433560	alice@example.com	False
3	tomcat	Tom Williams	1995-01-03	2025-11-15 20:45:21.433560	tom@example.com	True
*/I should count the number of users in the users table.
Action: sql_db_query
Action Input: SELECT c

#### Executing Agent & available tools #####

In [54]:


from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

tools = [search_documents, search_database]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")]
)

agent = create_tool_calling_agent(openai_llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

res = agent_executor.invoke({"input": "How many users are there in the table?"})

res



> Entering new AgentExecutor chain...

Invoking: `search_database` with `{'input': 'How many users are there in the table?'}`




> Entering new AgentExecutor chain...

Invoking: `search_database` with `{'input': 'How many users are there in the table?'}`




> Entering new AgentExecutor chain...

Invoking: `search_database` with `{'input': 'How many users are there in the table?'}`




> Entering new AgentExecutor chain...

Invoking: `search_database` with `{'input': 'How many users are there in the table?'}`




> Entering new AgentExecutor chain...

Invoking: `search_database` with `{'input': 'How many users are there in the table?'}`




> Entering new AgentExecutor chain...


KeyboardInterrupt: 

##### Database agent - using SQLDatabaseToolkit

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

toolkit = SQLDatabaseToolkit(db=db, llm=google_llm)

tools = toolkit.get_tools()
context = toolkit.get_context() # print(tools, "\n\n\n", context)

# Create custom prompt
messages = [
    ("system", """You are a helpful SQL assistant. 
        Answer questions about the database.
        
        Available tables: {table_names}
        Database Schema: {table_info}
        
     """),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
]

prompt = ChatPromptTemplate.from_messages(messages)
prompt = prompt.partial(**context)

agent = create_tool_calling_agent(google_llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

res = agent_executor.invoke({"input": "What is the email of samantha lee"})

res
